In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [6]:
%%writefile aisugst.py
#!/usr/bin/env python3
"""
Complete Classical Image Processing Pipeline for Egg Segmentation.
Incorporates:
  1. Bilateral edge-preserving filtering
  2. Multi-space chrominance thresholding (YCrCb skin suppression + Lab/HSV)
  3. Border component clearance (removes table edges, clothing, and walls)
  4. Morphological opening and closing
  5. Distance transform with multi-tier adaptive seed extraction
  6. Marker-controlled watershed to separate touching eggs
  7. Strict geometric validation (Scale, Solidity, Aspect Ratio, Circularity)
"""

import argparse
from pathlib import Path
import cv2
import numpy as np

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}


def clear_border_components(binary_mask: np.ndarray, border_margin: int = 4) -> np.ndarray:
    """Eliminates any foreground component touching the image boundaries."""
    H, W = binary_mask.shape[:2]
    cleared = binary_mask.copy()
    contours, _ = cv2.findContours(cleared, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if x <= border_margin or y <= border_margin or (x + w) >= (W - border_margin) or (y + h) >= (H - border_margin):
            cv2.drawContours(cleared, [cnt], -1, 0, thickness=cv2.FILLED)

    return cleared


def build_candidate_mask(img_bgr: np.ndarray) -> np.ndarray:
    """
    Separates eggshell reflectance from hand skin and background clutter
    via cross-color space analysis (YCrCb, HSV, L*a*b*).
    """
    H, W = img_bgr.shape[:2]
    img_area = H * W

    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

    _, Cr, Cb = cv2.split(ycrcb)
    _, S, _ = cv2.split(hsv)
    L, a, b = cv2.split(lab)

    # 1. Skin suppression locus (standard YCrCb human skin boundaries)
    skin_mask = cv2.inRange(ycrcb, np.array([0, 133, 77]), np.array([255, 175, 127]))

    # 2. White Egg Profile: High Lightness, Low Saturation, Non-Skin
    _, l_thresh = cv2.threshold(L, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, s_inv = cv2.threshold(S, 80, 255, cv2.THRESH_BINARY_INV)

    white_egg_cand = cv2.bitwise_and(l_thresh, s_inv)
    white_egg_cand = cv2.bitwise_and(white_egg_cand, cv2.bitwise_not(skin_mask))

    # 3. Brown Egg Profile: Yellowish chrominance separation (b* - a*)
    diff = cv2.subtract(b, a)
    _, brown_egg_cand = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    brown_egg_cand = cv2.bitwise_and(brown_egg_cand, cv2.bitwise_not(skin_mask))

    # Check which color distribution captures plausible egg area
    if np.sum(white_egg_cand > 0) >= 0.005 * img_area:
        candidate = white_egg_cand
    elif np.sum(brown_egg_cand > 0) >= 0.005 * img_area:
        candidate = brown_egg_cand
    else:
        # Fallback if both chromatic filters over-suppressed
        candidate = cv2.bitwise_and(l_thresh, cv2.bitwise_not(skin_mask))

    return candidate


def segment_eggs_classical(image_bgr: np.ndarray) -> np.ndarray:
    H, W = image_bgr.shape[:2]
    img_area = H * W

    # 1. Edge-preserving smoothing
    smoothed = cv2.bilateralFilter(image_bgr, d=7, sigmaColor=60, sigmaSpace=60)

    # 2. Extract chromatic candidate mask
    candidate = build_candidate_mask(smoothed)

    # 3. Morphological opening to cut thin skin bridges and hair
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    
    cleaned = cv2.morphologyEx(candidate, cv2.MORPH_OPEN, k_open, iterations=2)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, k_close, iterations=1)

    # 4. Remove elements touching image borders
    cleaned = clear_border_components(cleaned, border_margin=5)

    if np.sum(cleaned > 0) < 0.003 * img_area:
        return np.zeros((H, W), dtype=np.uint8)

    # 5. Sure Background Area
    sure_bg = cv2.dilate(cleaned, k_open, iterations=3)

    # 6. Euclidean Distance Transform
    dist = cv2.distanceTransform(cleaned, cv2.DIST_L2, 5)
    max_dist = dist.max()

    if max_dist <= 0:
        return np.zeros((H, W), dtype=np.uint8)

    # 7. Adaptive Multi-Tier Foreground Seed Extraction
    sure_fg = np.zeros((H, W), dtype=np.uint8)
    for ratio in [0.45, 0.32, 0.20]:
        _, sure_fg = cv2.threshold(dist, ratio * max_dist, 255, cv2.THRESH_BINARY)
        sure_fg = np.uint8(sure_fg)
        num_markers, _ = cv2.connectedComponents(sure_fg)
        if num_markers > 1:
            break

    # 8. Marker Formulation & Watershed
    unknown = cv2.subtract(sure_bg, sure_fg)
    num_markers, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    cv2.watershed(smoothed, markers)

    # 9. Morphological & Geometric Validation
    final_mask = np.zeros((H, W), dtype=np.uint8)

    for label in range(2, num_markers + 1):
        comp = (markers == label).astype(np.uint8) * 255
        comp = cv2.morphologyEx(comp, cv2.MORPH_CLOSE, k_open)

        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)

        # Scale Constraint: 0.6% to 45% of total frame
        if not (0.006 * img_area < area < 0.45 * img_area):
            continue

        # Solidity Constraint: Area / Convex Hull Area
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        if hull_area <= 0:
            continue
        solidity = float(area) / hull_area
        if solidity < 0.84:
            continue

        # Circularity / Form Factor: 4 * pi * Area / Perimeter^2
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        circularity = 4 * np.pi * (area / (perimeter * perimeter))
        if circularity < 0.45:
            continue

        # Aspect Ratio via Fitted Ellipse
        if len(cnt) >= 5:
            (_, _), (d1, d2), _ = cv2.fitEllipse(cnt)
            major = max(d1, d2)
            minor = min(d1, d2)
            aspect_ratio = major / (minor + 1e-5)
            if not (1.01 <= aspect_ratio <= 1.90):
                continue

        # Valid egg instance confirmed
        cv2.drawContours(final_mask, [cnt], -1, 255, thickness=cv2.FILLED)

    return final_mask


def main():
    parser = argparse.ArgumentParser(
        description="Classical Image Processing Pipeline for Egg Segmentation."
    )
    parser.add_argument("--input_dir", type=str, required=True, help="Path to input images folder")
    parser.add_argument("--output_dir", type=str, required=True, help="Path to save output binary masks")
    args = parser.parse_args()

    input_path = Path(args.input_dir)
    output_path = Path(args.output_dir)

    if not input_path.is_dir():
        raise FileNotFoundError(f"Directory not found: {input_path}")

    output_path.mkdir(parents=True, exist_ok=True)

    image_files = sorted(
        [p for p in input_path.iterdir() if p.suffix.lower() in VALID_EXTENSIONS]
    )

    if not image_files:
        print(f"No image files found in {input_path}")
        return

    print(f"Found {len(image_files)} images. Executing classical segmentation pipeline...")

    for idx, path in enumerate(image_files, 1):
        img_bgr = cv2.imread(str(path))
        if img_bgr is None:
            continue

        mask = segment_eggs_classical(img_bgr)
        out_name = output_path / f"{path.stem}_mask.png"
        cv2.imwrite(str(out_name), mask)

        if idx % 10 == 0 or idx == len(image_files):
            print(f"[{idx}/{len(image_files)}] Processed: {path.name}")

    print(f"\nPipeline finished. Segmented masks stored at: {output_path.resolve()}")


if __name__ == "__main__":
    main()

Overwriting aisugst.py


In [6]:
%%writefile segment_eggs_prompted.py
#!/usr/bin/env python3
import argparse
from pathlib import Path
import cv2
import numpy as np
import torch
from PIL import Image
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from ultralytics import FastSAM

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}

def get_egg_bboxes(image_pil: Image.Image, processor, detector, device: str, score_threshold: float = 0.22):
    """
    Open-vocabulary detection to isolate egg bounding boxes and suppress hands/background.
    """
    texts = [["an egg", "a white egg", "a brown egg"]]
    inputs = processor(text=texts, images=image_pil, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = detector(**inputs)

    target_sizes = torch.tensor([image_pil.size[::-1]]).to(device)
    results = processor.post_process_grounded_object_detection(
    outputs=outputs,
    threshold=score_threshold,
    text_labels=texts,
    target_sizes=target_sizes,
)[0]
    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()

    if len(boxes) == 0:
        return []

    # Non-Maximum Suppression across overlapping prompt detections
    keep_indices = cv2.dnn.NMSBoxes(
        bboxes=[[int(b[0]), int(b[1]), int(b[2] - b[0]), int(b[3] - b[1])] for b in boxes],
        scores=scores.tolist(),
        score_threshold=score_threshold,
        nms_threshold=0.45
    )

    if len(keep_indices) == 0:
        return []

    return [boxes[i] for i in keep_indices.flatten()]


def segment_egg_bboxes(image_path: Path, bboxes: list, sam_model: FastSAM, device: str) -> np.ndarray | None:
    """
    Run FastSAM using the detected bounding boxes as precise prompts.
    """
    img_bgr = cv2.imread(str(image_path))
    H, W = img_bgr.shape[:2]
    combined_mask = np.zeros((H, W), dtype=np.uint8)

    # Run inference on image
    results = sam_model(
        str(image_path),
        device=device,
        retina_masks=True,
        imgsz=1024,
        conf=0.25,
        verbose=False,
    )

    if results[0].masks is None:
        return None

    raw_masks = results[0].masks.data.cpu().numpy()  # [N, H, W]

    for bbox in bboxes:
        x1, y1, x2, y2 = bbox
        box_area = (x2 - x1) * (y2 - y1)
        
        best_mask = None
        best_overlap_score = 0.0

        # Match FastSAM masks to the OWLv2 bounding box
        for m in raw_masks:
            m_binary = (m > 0).astype(np.uint8)
            mask_area = np.sum(m_binary)
            if mask_area == 0:
                continue

            # Intersection between mask and the bounding box ROI
            roi_slice = m_binary[int(y1):int(y2), int(x1):int(x2)]
            in_box_area = np.sum(roi_slice)

            # Mask must reside primarily inside the detected box, not flood outside (e.g. into palm)
            precision = in_box_area / mask_area
            recall = in_box_area / (box_area + 1e-5)
            
            # Prioritize high concentration inside box
            if precision > 0.70 and in_box_area > 0:
                score = 2 * (precision * recall) / (precision + recall + 1e-5)
                if score > best_overlap_score:
                    best_overlap_score = score
                    best_mask = m_binary

        if best_mask is not None:
            combined_mask = np.bitwise_or(combined_mask, best_mask * 255)

    return combined_mask if np.any(combined_mask) else None


def main():
    parser = argparse.ArgumentParser(description="Zero-shot prompt guided egg segmentation.")
    parser.add_argument("--input_dir", type=str, required=True, help="Input directory")
    parser.add_argument("--output_dir", type=str, required=True, help="Output directory")
    args = parser.parse_args()

    input_dir = Path(args.input_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted([p for p in input_dir.iterdir() if p.suffix.lower() in VALID_EXTENSIONS])
    if not image_paths:
        print("No valid images found.")
        return

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    # Load zero-shot detector
    print("Loading OWLv2 Detector...")
    processor = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
    detector = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble").to(device)
    detector.eval()

    # Load FastSAM
    print("Loading FastSAM-x...")
    sam_model = FastSAM("FastSAM-x.pt")

    for idx, img_path in enumerate(image_paths, 1):
        print(f"[{idx}/{len(image_paths)}] Processing {img_path.name}...", end=" ")
        
        pil_img = Image.open(img_path).convert("RGB")
        bboxes = get_egg_bboxes(pil_img, processor, detector, device)

        if not bboxes:
            print("No egg bbox detected.")
            continue

        mask = segment_egg_bboxes(img_path, bboxes, sam_model, device)

        if mask is not None:
            out_path = output_dir / f"{img_path.stem}_mask.png"
            cv2.imwrite(str(out_path), mask)
            print("Mask saved.")
        else:
            print("Mask extraction failed.")

    print(f"\nCompleted. Clean ground-truth masks written to {output_dir}")

if __name__ == "__main__":
    main()

Overwriting segment_eggs_prompted.py


In [4]:
%%writefile datasetSpliter.py
#!/usr/bin/env python3


"""
Dataset Splitter Script (4:1 Dev/Eval Split).
Splits matched pairs from 'Eggs' and 'Masks' folders while preventing data leakage.
"""

import argparse
import os
from pathlib import Path
import random
import re
import shutil

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}


def extract_image_index(filename: str) -> int | None:
  """Extracts the integer index 'i' from titles like 'Not Damaged (i)'."""
  match = re.search(r"Not\s*Damaged\s*\((\d+)\)", filename, re.IGNORECASE)
  if match:
    return int(match.group(1))
  return None


def split_dataset(
    eggs_dir: Path, masks_dir: Path, output_dir: Path, seed: int = 42
):
  egg_files = [
      p
      for p in eggs_dir.iterdir()
      if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
  ]
  mask_files = [
      p
      for p in masks_dir.iterdir()
      if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
  ]

  # Map index -> Path
  egg_map = {}
  for p in egg_files:
    idx = extract_image_index(p.stem)
    if idx is not None:
      egg_map[idx] = p

  mask_map = {}
  for p in mask_files:
    idx = extract_image_index(p.stem)
    if idx is not None:
      mask_map[idx] = p

  # Pair on identical numeric indices
  common_indices = sorted(list(set(egg_map.keys()) & set(mask_map.keys())))

  if not common_indices:
    print(
        "Error: No matching pairs of 'Not Damaged (i)' images and masks found."
    )
    return

  print(f"Total matching pairs found: {len(common_indices)}")

  # Deterministic shuffle to ensure reproducibility across runs
  rng = random.Random(seed)
  shuffled_indices = common_indices.copy()
  rng.shuffle(shuffled_indices)

  # 4:1 Split (80% Dev, 20% Eval)
  num_total = len(shuffled_indices)
  num_eval = int(round(num_total * 0.20))
  num_dev = num_total - num_eval

  dev_indices = shuffled_indices[:num_dev]
  eval_indices = shuffled_indices[num_dev:]

  print(
      f"Split breakdown: Dev = {len(dev_indices)}"
      f" ({len(dev_indices)/num_total*100:.1f}%), Eval = {len(eval_indices)}"
      f" ({len(eval_indices)/num_total*100:.1f}%)"
  )

  splits = {"dev": dev_indices, "eval": eval_indices}

  for split_name, indices in splits.items():
    split_eggs_dir = output_dir / split_name / "Eggs"
    split_masks_dir = output_dir / split_name / "Masks"
    split_eggs_dir.mkdir(parents=True, exist_ok=True)
    split_masks_dir.mkdir(parents=True, exist_ok=True)

    for idx in indices:
      egg_src = egg_map[idx]
      mask_src = mask_map[idx]

      shutil.copy2(egg_src, split_eggs_dir / egg_src.name)
      shutil.copy2(mask_src, split_masks_dir / mask_src.name)

  print(f"Dataset successfully split and saved to: {output_dir.resolve()}")


def main():
  parser = argparse.ArgumentParser(
      description="Split dataset into 4:1 Dev/Eval splits."
  )
  parser.add_argument(
      "--eggs_dir",
      type=str,
      default="Eggs",
      help="Path to source Eggs directory",
  )
  parser.add_argument(
      "--masks_dir",
      type=str,
      default="Masks",
      help="Path to source Masks directory",
  )
  parser.add_argument(
      "--output_dir",
      type=str,
      default="dataset_split",
      help="Target directory for output splits",
  )
  parser.add_argument(
      "--seed",
      type=int,
      default=42,
      help="Random seed for deterministic split",
  )
  args = parser.parse_args()

  split_dataset(
      Path(args.eggs_dir),
      Path(args.masks_dir),
      Path(args.output_dir),
      seed=args.seed,
  )


if __name__ == "__main__":
  main()

Overwriting datasetSpliter.py


In [6]:
%%writefile cannyOtsuGeom.py
#!/usr/bin/env python3
"""
Baseline Classical Segmentation Pipeline for Task 2.
Combines Canny Edge Detection, Otsu Thresholding, Morphological Operations,
and Geometric Constraints (Area, Solidity, Aspect Ratio).
"""

import argparse
from pathlib import Path
import cv2
import numpy as np

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}


def segment_egg_classical(image_bgr: np.ndarray) -> np.ndarray:
    H, W = image_bgr.shape[:2]
    img_area = H * W

    # 1. Preprocessing & Color conversion
    blurred = cv2.GaussianBlur(image_bgr, (5, 5), 0)
    gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)

    # 2. Otsu Thresholding (Region Proposal)
    _, otsu_mask = cv2.threshold(
        gray, 0, 255, cv2.THRESH_OTSU
    )

    # Invert if the background is brighter than the object
    if np.mean(otsu_mask) > 127:
        otsu_mask = cv2.bitwise_not(otsu_mask)

    # 3. Canny Edge Detection
    # Derive adaptive threshold bounds based on median intensity
    median_val = np.median(gray)
    lower_thresh = int(max(0, (1.0 - 0.33) * median_val))
    upper_thresh = int(min(255, (1.0 + 0.33) * median_val))
    edges = cv2.Canny(gray, lower_thresh, upper_thresh)

    # 4. Morphological Boundary Closing
    kernel_edge = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    closed_edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel_edge)

    # Fill closed contours formed by Canny
    edge_filled = np.zeros((H, W), dtype=np.uint8)
    contours_edge, _ = cv2.findContours(
        closed_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    for c in contours_edge:
        if cv2.contourArea(c) > 0.005 * img_area:
            cv2.drawContours(edge_filled, [c], -1, 255, thickness=cv2.FILLED)

    # 5. Fusion: Constrain Otsu region proposal with Canny edge containment
    combined_raw = cv2.bitwise_or(otsu_mask, edge_filled)

    # Morphological Clean-up
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    cleaned = cv2.morphologyEx(combined_raw, cv2.MORPH_OPEN, kernel_clean)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_clean)

    # 6. Geometric Property Filtering
    final_mask = np.zeros((H, W), dtype=np.uint8)
    contours, _ = cv2.findContours(
        cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Scale Constraint: 0.5% to 40% of image size
        if not (0.005 * img_area < area < 0.40 * img_area):
            continue

        # Solidity Constraint: Area / Convex Hull Area
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        if hull_area == 0:
            continue
        solidity = float(area) / hull_area
        if solidity < 0.85:
            continue

        # Circularity / Form Factor: 4 * pi * Area / Perimeter^2
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        circularity = 4 * np.pi * (area / (perimeter * perimeter))
        if not (0.55 <= circularity <= 0.95):
            continue

        # Aspect Ratio via Fitted Ellipse
        if len(cnt) >= 5:
            (_, _), (major_axis, minor_axis), _ = cv2.fitEllipse(cnt)
            aspect_ratio = max(major_axis, minor_axis) / (
                min(major_axis, minor_axis) + 1e-5
            )
            if not (1.05 <= aspect_ratio <= 1.85):
                continue
        else:
            continue

        # Valid candidate: write to final mask
        cv2.drawContours(final_mask, [cnt], -1, 255, thickness=cv2.FILLED)

    return final_mask


def main():
    parser = argparse.ArgumentParser(
        description="Run Canny + Otsu + Aspect Ratio Classical Egg Segmentation."
    )
    parser.add_argument(
        "--input_dir",
        type=str,
        required=True,
        help="Path to development/eval split images",
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        required=True,
        help="Path to save output binary masks",
    )
    args = parser.parse_args()

    input_path = Path(args.input_dir)
    output_path = Path(args.output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    img_files = sorted(
        [p for p in input_path.iterdir() if p.suffix.lower() in VALID_EXTENSIONS]
    )
    if not img_files:
        print(f"No valid image files located in {input_path}")
        return

    print(f"Processing {len(img_files)} images via classical edge-otsu fusion...")
    for idx, path in enumerate(img_files, 1):
        img_bgr = cv2.imread(str(path))
        if img_bgr is None:
            continue

        mask = segment_egg_classical(img_bgr)
        out_name = output_path / f"{path.stem}_classical_mask.png"
        cv2.imwrite(str(out_name), mask)

        if idx % 10 == 0 or idx == len(img_files):
            print(f"[{idx}/{len(img_files)}] Processed {path.name}")

    print(f"Classical inference finished. Output masks stored in: {output_path}")


if __name__ == "__main__":
    main()

Writing cannyOtsuGeom.py


In [14]:
%%writefile watershed.py
#!/usr/bin/env python3
"""
Trial 3: Marker-Controlled Distance-Transform Watershed Segmentation.
Designed for Task 2 to separate touching eggs and clean boundary contacts.
"""

import argparse
from pathlib import Path
import cv2
import numpy as np

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}


def segment_egg_watershed(image_bgr: np.ndarray) -> np.ndarray:
    H, W = image_bgr.shape[:2]
    img_area = H * W

    # 1. Edge-preserving smoothing
    smoothed = cv2.bilateralFilter(image_bgr, d=7, sigmaColor=50, sigmaSpace=50)

    # 2. Color-space candidate masking (Separating shell from hand skin)
    lab = cv2.cvtColor(smoothed, cv2.COLOR_BGR2LAB)
    hsv = cv2.cvtColor(smoothed, cv2.COLOR_BGR2HSV)

    L, a, b = cv2.split(lab)
    _, S, _ = cv2.split(hsv)

    # White egg prompt: High Luminance + Low Saturation
    _, l_mask = cv2.threshold(L, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, s_inv_mask = cv2.threshold(S, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    candidate = cv2.bitwise_and(l_mask, s_inv_mask)

    # Brown egg fallback (b* - a* difference)
    if np.sum(candidate > 0) < 0.005 * img_area:
        diff = cv2.subtract(b, a)
        _, candidate = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. Morphological opening to eliminate fine finger edges & creases
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    opening = cv2.morphologyEx(candidate, cv2.MORPH_OPEN, kernel, iterations=2)

    # 4. Sure Background Area (Dilate candidate mask)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)

    # 5. Sure Foreground Area via Euclidean Distance Transform
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    max_val = dist_transform.max()
    if max_val <= 0:
        return np.zeros((H, W), dtype=np.uint8)

    # Threshold distance transform to locate distinct egg core centers
    _, sure_fg = cv2.threshold(dist_transform, 0.42 * max_val, 255, cv2.THRESH_BINARY)
    sure_fg = np.uint8(sure_fg)

    # 6. Unknown Boundary Region (between sure foreground and sure background)
    unknown = cv2.subtract(sure_bg, sure_fg)

    # 7. Marker Labeling
    num_markers, markers = cv2.connectedComponents(sure_fg)

    if num_markers <= 1:
        # Fallback if distance thresholding eliminated weak detections
        _, sure_fg = cv2.threshold(dist_transform, 0.25 * max_val, 255, cv2.THRESH_BINARY)
        sure_fg = np.uint8(sure_fg)
        unknown = cv2.subtract(sure_bg, sure_fg)
        num_markers, markers = cv2.connectedComponents(sure_fg)

    # Shift markers so background is 1, unknown is 0
    markers = markers + 1
    markers[unknown == 255] = 0

    # 8. Run Watershed
    # Watershed expects a 3-channel BGR image to compute topographic gradients
    cv2.watershed(smoothed, markers)

    # 9. Extract and Filter Individual Segmented Egg Components
    final_mask = np.zeros((H, W), dtype=np.uint8)

    # Marker indices 2..num_markers correspond to foreground objects
    for label in range(2, num_markers + 1):
        component_mask = (markers == label).astype(np.uint8) * 255
        
        # Morphological close to bridge boundary pixels (-1 watershed ridge lines)
        close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        component_mask = cv2.morphologyEx(component_mask, cv2.MORPH_CLOSE, close_kernel)

        contours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)

        # Scale Constraint: 0.8% to 45% of total frame
        if not (0.008 * img_area < area < 0.45 * img_area):
            continue

        # Solidity Constraint (Eggs are highly convex ovoids)
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        if hull_area <= 0:
            continue
        solidity = float(area) / hull_area
        if solidity < 0.82:
            continue

        # Aspect Ratio Validation
        if len(cnt) >= 5:
            (_, _), (d1, d2), _ = cv2.fitEllipse(cnt)
            major = max(d1, d2)
            minor = min(d1, d2)
            aspect_ratio = major / (minor + 1e-5)
            if not (1.02 <= aspect_ratio <= 1.95):
                continue

        # Merge validated egg mask
        cv2.drawContours(final_mask, [cnt], -1, 255, thickness=cv2.FILLED)

    return final_mask


def main():
    parser = argparse.ArgumentParser(
        description="Run Marker-Controlled Watershed Egg Segmentation."
    )
    parser.add_argument("--input_dir", type=str, required=True, help="Input images folder")
    parser.add_argument("--output_dir", type=str, required=True, help="Output masks folder")
    args = parser.parse_args()

    input_dir = Path(args.input_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted([p for p in input_dir.iterdir() if p.suffix.lower() in VALID_EXTENSIONS])
    print(f"Running Watershed segmentation on {len(image_paths)} images...")

    for idx, p in enumerate(image_paths, 1):
        img = cv2.imread(str(p))
        if img is None:
            continue
        mask = segment_egg_watershed(img)
        out_file = output_dir / f"{p.stem}_watershed_mask.png"
        cv2.imwrite(str(out_file), mask)

        if idx % 10 == 0 or idx == len(image_paths):
            print(f"[{idx}/{len(image_paths)}] Processed {p.name}")

    print(f"Processing complete. Masks stored in: {output_dir}")


if __name__ == "__main__":
    main()

Writing watershed.py


In [14]:
%%writefile comparison.py
#!/usr/bin/env python3
"""
Task 3: Comparative Quantitative Evaluation Script.
Computes Mean IoU and Pixel Accuracy across three classical pipelines
against Ground Truth (Task 1) masks, producing publication-ready styled tables
saved via Matplotlib.
"""

import argparse
import os
from pathlib import Path
import re
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}


def extract_image_index(filename: str) -> int | None:
  """Extracts numeric index 'i' from titles like 'Not Damaged (i)...'"""
  match = re.search(r"Not\s*Damaged\s*\((\d+)\)", filename, re.IGNORECASE)
  if match:
    return int(match.group(1))
  return None


def index_directory(dir_path: Path) -> dict[int, Path]:
  """Maps integer index to file path for all valid mask files in directory."""
  mapping = {}
  if not dir_path.is_dir():
    return mapping
  for p in dir_path.iterdir():
    if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS:
      idx = extract_image_index(p.stem)
      if idx is not None:
        mapping[idx] = p
  return mapping


def compute_metrics(pred_bin: np.ndarray, gt_bin: np.ndarray):
  """Computes Intersection-over-Union (IoU), Pixel Accuracy, and Dice (F1).

  pred_bin and gt_bin are binary boolean arrays (True for foreground).
  """
  intersection = np.logical_and(pred_bin, gt_bin).sum()
  union = np.logical_or(pred_bin, gt_bin).sum()

  # Standard IoU definition with empty-empty handling
  if union == 0:
    iou = 1.0 if intersection == 0 else 0.0
  else:
    iou = float(intersection) / float(union)

  # Pixel Accuracy: (TP + TN) / Total
  total_pixels = pred_bin.size
  correct_pixels = (pred_bin == gt_bin).sum()
  pixel_acc = float(correct_pixels) / float(total_pixels)

  # Dice / F1 Score
  denom = pred_bin.sum() + gt_bin.sum()
  dice = (2.0 * intersection) / denom if denom > 0 else 1.0

  return iou, pixel_acc, dice


def render_table(
    df: pd.DataFrame,
    title: str,
    output_path: Path,
    highlight_col: str = None,
    ascending: bool = False,
):
  """Renders and saves a clean, publication-ready table graphic using Matplotlib."""
  fig, ax = plt.subplots(figsize=(10, len(df) * 0.7 + 1.8), dpi=300)
  ax.axis("off")
  ax.axis("tight")

  col_labels = list(df.columns)
  cell_text = []
  cell_colors = []

  df_sorted = (
      df.sort_values(by=highlight_col, ascending=ascending)
      if highlight_col
      else df
  )

  for _, row in df_sorted.iterrows():
    row_text = []
    row_colors = []
    for col in col_labels:
      val = row[col]
      if isinstance(val, float):
        row_text.append(f"{val:.4f}")
      else:
        row_text.append(str(val))
      row_colors.append("#F8F9FA")
    cell_text.append(row_text)
    cell_colors.append(row_colors)

  table = ax.table(
      cellText=cell_text,
      colLabels=col_labels,
      cellColours=cell_colors,
      loc="center",
      cellLoc="center",
  )

  table.auto_set_font_size(False)
  table.set_fontsize(11)
  table.scale(1.2, 1.8)

  # Header styling
  for (row_idx, _), cell in table.get_celld().items():
    if row_idx == 0:
      cell.set_facecolor("#2B3A42")
      cell.set_text_props(color="white", weight="bold")
    else:
      cell.set_edgecolor("#DEE2E6")

  plt.title(title, fontsize=14, weight="bold", pad=15, color="#1A252C")
  plt.tight_layout()
  plt.savefig(output_path, bbox_inches="tight", dpi=300)
  plt.close()
  print(f"Saved table image: {output_path}")


def main():
  parser = argparse.ArgumentParser(
      description="Compute IoU & Pixel Accuracy across 3 Classical Methods vs"
      " GT."
  )
  parser.add_argument(
      "--gt_dir",
      type=str,
      required=True,
      help="Task 1 Ground Truth Masks dir",
  )
  parser.add_argument(
      "--otsucanny_dir", type=str, required=True, help="Otsu+Canny Masks dir"
  )
  parser.add_argument(
      "--watershed_dir",
      type=str,
      required=True,
      help="Standard Watershed Masks dir",
  )
  parser.add_argument(
      "--pipeline_dir",
      type=str,
      required=True,
      help="Suggested Refined Pipeline Masks dir",
  )
  parser.add_argument(
      "--output_dir",
      type=str,
      default="./eval_metrics",
      help="Directory for output tables",
  )
  args = parser.parse_args()

  gt_dir = Path(args.gt_dir)
  oc_dir = Path(args.otsucanny_dir)
  ws_dir = Path(args.watershed_dir)
  pl_dir = Path(args.pipeline_dir)
  out_dir = Path(args.output_dir)
  out_dir.mkdir(parents=True, exist_ok=True)

  print("Indexing directories...")
  gt_map = index_directory(gt_dir)
  oc_map = index_directory(oc_dir)
  ws_map = index_directory(ws_dir)
  pl_map = index_directory(pl_dir)

  common_indices = sorted(
      list(
          set(gt_map.keys())
          & (set(oc_map.keys()) | set(ws_map.keys()) | set(pl_map.keys()))
      )
  )
  print(
      f"Found {len(common_indices)} paired evaluation samples with Ground"
      " Truth."
  )

  methods = {
      "Trial 1: Otsu + Canny": oc_map,
      "Trial 2: Watershed": ws_map,
      "Trial 3: AI-Watershed Refined": pl_map,
  }

  records = []

  for idx in common_indices:
    gt_path = gt_map[idx]
    gt_img = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
    if gt_img is None:
      continue
    gt_bin = gt_img > 127
    H, W = gt_bin.shape

    for name, m_map in methods.items():
      if idx in m_map:
        pred_img = cv2.imread(str(m_map[idx]), cv2.IMREAD_GRAYSCALE)
        if pred_img is not None:
          if pred_img.shape != (H, W):
            pred_img = cv2.resize(
                pred_img, (W, H), interpolation=cv2.INTER_NEAREST
            )
          pred_bin = pred_img > 127
        else:
          pred_bin = np.zeros_like(gt_bin)
      else:
        pred_bin = np.zeros_like(gt_bin)

      iou, pix_acc, dice = compute_metrics(pred_bin, gt_bin)
      records.append({
          "Sample_ID": idx,
          "Method": name,
          "IoU": iou,
          "Pixel_Accuracy": pix_acc,
          "Dice": dice,
      })

  df_all = pd.DataFrame(records)

  # -------------------------------------------------------------
  # Table 1: Global Summary Metrics (Mean IoU, Dice, Pixel Acc)
  # -------------------------------------------------------------
  summary_list = []
  for name in methods.keys():
    sub = df_all[df_all["Method"] == name]
    summary_list.append({
        "Method / Trial Pipeline": name,
        "Mean IoU (Jaccard)": sub["IoU"].mean(),
        "Std IoU": sub["IoU"].std(),
        "Mean Pixel Accuracy": sub["Pixel_Accuracy"].mean(),
        "Mean Dice (F1)": sub["Dice"].mean(),
    })

  df_summary = pd.DataFrame(summary_list)
  df_summary.to_csv(out_dir / "summary_metrics.csv", index=False)
  print("\n=== Global Evaluation Summary ===")
  print(df_summary.to_string(index=False))

  render_table(
      df_summary,
      title=(
          "Table 1: Quantitative Segmentation Performance vs Ground Truth"
          " (Task 1)"
      ),
      output_path=out_dir / "table1_overall_iou_metrics.png",
      highlight_col="Mean IoU (Jaccard)",
      ascending=False,
  )

  # -------------------------------------------------------------
  # Table 2: Pixel Accuracy Distribution & Dropout Diagnostics
  # -------------------------------------------------------------
  breakdown_list = []
  for name in methods.keys():
    sub = df_all[df_all["Method"] == name]
    total = len(sub)
    dropouts = (sub["IoU"] == 0.0).sum()
    high_quality = (sub["IoU"] >= 0.70).sum()

    breakdown_list.append({
        "Method Pipeline": name,
        "Mean Pixel Acc (%)": sub["Pixel_Accuracy"].mean() * 100.0,
        "Median Pixel Acc (%)": sub["Pixel_Accuracy"].median() * 100.0,
        "Min Pixel Acc (%)": sub["Pixel_Accuracy"].min() * 100.0,
        "Complete Dropouts (IoU=0)": (
            f"{dropouts} / {total} ({dropouts/total*100:.1f}%)"
        ),
        "High Overlap (IoU ≥ 0.70)": (
            f"{high_quality} / {total} ({high_quality/total*100:.1f}%)"
        ),
    })

  df_breakdown = pd.DataFrame(breakdown_list)
  df_breakdown.to_csv(out_dir / "pixel_accuracy_breakdown.csv", index=False)
  print("\n=== Pixel Accuracy & Failure Distribution ===")
  print(df_breakdown.to_string(index=False))

  render_table(
      df_breakdown,
      title="Table 2: Pixel Accuracy Distribution & Dropout Diagnostics",
      output_path=out_dir / "table2_pixel_accuracy_breakdown.png",
  )

  print(f"\nAll tables generated successfully in: {out_dir.resolve()}")


if __name__ == "__main__":
  main()

Overwriting comparison.py


In [15]:
!python comparison.py \
    --gt_dir "/kaggle/working/reduced/dev/Masks" \
    --otsucanny_dir "/kaggle/working/CannyOtsuGeomMasks" \
    --watershed_dir "/kaggle/working/WatershedMasks" \
    --pipeline_dir "/kaggle/working/ClassicalMasks" \
    --output_dir "/kaggle/working/EvaluationOutputs"

Indexing directories...
Found 76 paired evaluation samples with Ground Truth.

=== Global Evaluation Summary ===
      Method / Trial Pipeline  Mean IoU (Jaccard)  Std IoU  Mean Pixel Accuracy  Mean Dice (F1)
        Trial 1: Otsu + Canny            0.057187 0.150407             0.564968        0.080494
           Trial 2: Watershed            0.254769 0.284327             0.641600        0.334279
Trial 3: AI-Watershed Refined            0.129107 0.235110             0.581480        0.171084
Saved table image: /kaggle/working/EvaluationOutputs/table1_overall_iou_metrics.png

=== Pixel Accuracy & Failure Distribution ===
              Method Pipeline  Mean Pixel Acc (%)  Median Pixel Acc (%)  Min Pixel Acc (%) Complete Dropouts (IoU=0) High Overlap (IoU ≥ 0.70)
        Trial 1: Otsu + Canny           56.496820             61.899658           8.198486           61 / 76 (80.3%)             1 / 76 (1.3%)
           Trial 2: Watershed           64.159989             71.078735           8.19

In [8]:
!python aisugst.py    --input_dir "/kaggle/working/reduced/dev/Eggs" --output_dir "/kaggle/working/ClassicalMasks"

Found 76 images. Executing classical segmentation pipeline...
[10/76] Processed: Not Damaged (11).jpg
[20/76] Processed: Not Damaged (29).jpg
[30/76] Processed: Not Damaged (43).jpg
[40/76] Processed: Not Damaged (56).jpg
[50/76] Processed: Not Damaged (67).jpg
[60/76] Processed: Not Damaged (77).jpg
[70/76] Processed: Not Damaged (91).jpg
[76/76] Processed: Not Damaged (99).jpg

Pipeline finished. Segmented masks stored at: /kaggle/working/ClassicalMasks


In [15]:
!python watershed.py \
    --input_dir "/kaggle/working/reduced/dev/Eggs" \
    --output_dir "/kaggle/working/WatershedMasks"

Running Watershed segmentation on 76 images...
[10/76] Processed Not Damaged (11).jpg
[20/76] Processed Not Damaged (29).jpg
[30/76] Processed Not Damaged (43).jpg
[40/76] Processed Not Damaged (56).jpg
[50/76] Processed Not Damaged (67).jpg
[60/76] Processed Not Damaged (77).jpg
[70/76] Processed Not Damaged (91).jpg
[76/76] Processed Not Damaged (99).jpg
Processing complete. Masks stored in: /kaggle/working/WatershedMasks


In [5]:
!python datasetSpliter.py --eggs_dir "/kaggle/input/datasets/shreyanshdas11/task-2/Task 2 Dataset/Eggs" --masks_dir "/kaggle/input/datasets/shreyanshdas11/task-2/Task 2 Dataset/Eggs" --output_dir "/kaggle/working/reduced" --seed 42

Total matching pairs found: 95
Split breakdown: Dev = 76 (80.0%), Eval = 19 (20.0%)
Dataset successfully split and saved to: /kaggle/working/reduced


In [12]:
!python cannyOtsuGeom.py  --input_dir "/kaggle/working/reduced/dev/Eggs"  --output_dir "/kaggle/working/CannyOtsuGeomMasks"

Processing 76 images via classical edge-otsu fusion...
[10/76] Processed Not Damaged (11).jpg
[20/76] Processed Not Damaged (29).jpg
[30/76] Processed Not Damaged (43).jpg
[40/76] Processed Not Damaged (56).jpg
[50/76] Processed Not Damaged (67).jpg
[60/76] Processed Not Damaged (77).jpg
[70/76] Processed Not Damaged (91).jpg
[76/76] Processed Not Damaged (99).jpg
Classical inference finished. Output masks stored in: /kaggle/working/CannyOtsuGeomMasks


In [11]:
!python segment_eggs_prompted.py \
    --input_dir "/kaggle/input/datasets/shreyanshdas11/eggs-dataset/Merged" \
    --output_dir "/kaggle/working/egg_masks"


python3: can't open file '/kaggle/working/segment_eggs_prompted.py': [Errno 2] No such file or directory


In [16]:
!zip -r results.zip /kaggle/working/EvaluationOutputs


  adding: kaggle/working/EvaluationOutputs/ (stored 0%)
  adding: kaggle/working/EvaluationOutputs/table2_pixel_accuracy_breakdown.png (deflated 11%)
  adding: kaggle/working/EvaluationOutputs/summary_metrics.csv (deflated 32%)
  adding: kaggle/working/EvaluationOutputs/table1_overall_iou_metrics.png (deflated 13%)
  adding: kaggle/working/EvaluationOutputs/pixel_accuracy_breakdown.csv (deflated 38%)
